# Chapter 12 &mdash; Disambiguation Measured: 1 Parse versus Many

**Concept 10 of the Chapter 12 decomposition:** *Disambiguation Measured: 1 Parse versus 36*

The layered E/T/F grammar yields a single computation; the ambiguous one yields many.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter12/Concept-Disambiguation-Measured/Concept-Disambiguation-Measured.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_PDA        import *
from jove.AnimatePDA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


Chapter 11 argued that layering removes ambiguity. Here it is **measured**: convert
both grammars to PDA and count accepting computations.

Each accepting computation of the goal machine corresponds to one **leftmost
derivation**, hence to one parse tree. So the PDA's computation count *is* the
ambiguity count &mdash; the abstract property becomes an observable number.

For the ambiguous expression grammar the count grows fast with the number of
operators; for the layered grammar it stays at **exactly 1**.

That is the practical argument for disambiguating: not elegance, but parser cost and
determinacy.

## 2. Definitions

### Both grammars, and the conversion

# --- a thin wrapper over Jove's PDA runner -----------------------------
# run_pda returns (surviving-IDs, accepting-paths, visited-IDs); a string
# is accepted exactly when the list of accepting paths is non-empty.
def pda_accepts(P, s, acceptance='ACCEPT_F', STKMAX=6):
    surv, paths, visited = run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)
    return len(paths) > 0

def pda_npaths(P, s, acceptance='ACCEPT_F', STKMAX=6):
    return len(run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)[1])

In [ ]:
# --- a tiny CFG toolkit -------------------------------------------------
# A grammar is a dict with keys N (nonterminals), Sigma (terminals),
# S (start symbol) and P (productions: nonterminal -> list of RHS tuples).
# A right-hand side is a tuple of one-character symbols; () is epsilon.
# By convention UPPERCASE single letters are nonterminals.

def mkg(rules, start='S'):
    N = set(rules)
    P = {A: [tuple(r) for r in rhs] for A, rhs in rules.items()}
    Sigma = {c for rhs in P.values() for r in rhs for c in r if c not in N}
    return dict(N=N, Sigma=Sigma, S=start, P=P)

def show(G):
    print("N     =", sorted(G['N']))
    print("Sigma =", sorted(G['Sigma']))
    print("S     =", G['S'])
    for A in sorted(G['P']):
        alts = ' | '.join((''.join(r) if r else "''") for r in G['P'][A])
        print("   %s -> %s" % (A, alts))

def derivable(G, maxlen):
    # least fixed point: for each nonterminal, every terminal string of
    # length <= maxlen it derives.  Far cheaper than searching sentential
    # forms, and it terminates because the sets only grow and are bounded.
    T = {A: set() for A in G['N']}
    def spans(r):
        acc = {''}
        for x in r:
            src = T[x] if x in T else {x}
            acc = {a + b for a in acc for b in src if len(a) + len(b) <= maxlen}
            if not acc: break
        return acc
    changed = True
    while changed:
        changed = False
        for A in G['P']:
            for r in G['P'][A]:
                for w in spans(r):
                    if w not in T[A]:
                        T[A].add(w); changed = True
    return T

def language(G, maxlen):
    return sorted(derivable(G, maxlen)[G['S']], key=lambda s: (len(s), s))

def _spans(G, w, cap=None):
    # Bottom-up, shortest span first, so a span never depends on a LONGER
    # one.  Within a span we iterate |N|+1 times, which is enough to close
    # unit rules (A -> B) and epsilon rules.  Doing it top-down with a
    # "cycle guard" silently poisons the memo table, so we do not.
    n, N, P = len(w), G['N'], G['P']
    tab = {}                       # (A, i, j) -> count, or list of trees
    def get(sym, i, j):
        if sym not in N:
            if j == i + 1 and w[i] == sym:
                return 1 if cap is None else [sym]
            return 0 if cap is None else []
        return tab.get((sym, i, j), 0 if cap is None else [])
    def seqv(r, i, j):
        if not r:
            if i != j: return 0 if cap is None else []
            return 1 if cap is None else [()]
        acc = 0 if cap is None else []
        for k in range(i, j + 1):
            a = get(r[0], i, k)
            if not a: continue
            b = seqv(r[1:], k, j)
            if not b: continue
            if cap is None:
                acc += a * b
            else:
                for h in a:
                    for t in b:
                        acc.append((h,) + tuple(t))
                        if len(acc) >= cap: return acc
        return acc
    for length in range(0, n + 1):
        for i in range(0, n - length + 1):
            j = i + length
            for _ in range(len(N) + 1):
                grew = False
                for A in P:
                    v = []
                    for r in P[A]:
                        x = seqv(r, i, j)
                        if cap is None:
                            v.append(x)
                        else:
                            v += [(A,) + tuple(t) for t in x]
                            if len(v) >= cap: v = v[:cap]; break
                    v = sum(v) if cap is None else v
                    old = tab.get((A, i, j), 0 if cap is None else [])
                    if (v != old) if cap is None else (len(v) != len(old)):
                        tab[(A, i, j)] = v; grew = True
                if not grew: break
    return get(G['S'], 0, n)

def nparses(G, w):
    return _spans(G, w, cap=None)

def parse_trees(G, w, cap=8):
    return _spans(G, w, cap=cap)

def yield_of(t):
    return t if isinstance(t, str) else ''.join(yield_of(c) for c in t[1:])

def show_tree(t, ind=0):
    if isinstance(t, str):
        print("%s'%s'" % ('  ' * ind, t)); return
    print("%s%s" % ('  ' * ind, t[0]))
    for c in t[1:]: show_tree(c, ind + 1)

def leftmost(G, w):
    # the leftmost derivation read off one parse tree
    ts = parse_trees(G, w, cap=1)
    if not ts: return None
    steps, form = [], [G['S']]
    def expand(t, pos):
        # t is the subtree rooted at the nonterminal currently at `pos`
        if isinstance(t, str): return pos + 1
        kids = [c if isinstance(c, str) else c[0] for c in t[1:]]
        form[pos:pos+1] = kids
        steps.append(''.join(form) or "''")
        p = pos
        for c in t[1:]:
            p = expand(c, p)
        return p
    steps.append(G['S'])
    expand(ts[0], 0)
    return steps


def cfg2pda(G):
    lines = ['PDA', "I : '' , # ; %s#  -> P" % G['S']]
    for A in sorted(G['P']):
        for r in G['P'][A]:
            push = ''.join(r) if r else "''"
            lines.append("P : '' , %s ; %-8s -> P" % (A, push))
    for a in sorted(G['Sigma']):
        lines.append("P : %s , %s ; ''  -> P" % (a, a))
    lines.append("P : '' , # ; #  -> F")
    return md2mc('\n'.join(lines))

Amb   = mkg({'E': ["1", "2", "3", "E+E", "E*E", "(E)"]}, 'E')
Layer = mkg({'E': ["T", "E+T"], 'T': ["F", "T*F"],
             'F': ["1", "2", "3", "(E)"]}, 'E')

# --- a thin wrapper over Jove's PDA runner -----------------------------
# run_pda returns (surviving-IDs, accepting-paths, visited-IDs); a string
# is accepted exactly when the list of accepting paths is non-empty.
def pda_accepts(P, s, acceptance='ACCEPT_F', STKMAX=6):
    surv, paths, visited = run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)
    return len(paths) > 0

def pda_npaths(P, s, acceptance='ACCEPT_F', STKMAX=6):
    return len(run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)[1])

## 3. Tests

Both grammars describe the same language.

In [ ]:
for n in [4, 5]:
    a, b = set(language(Amb, n)), set(language(Layer, n))
    print("up to length %d : %d strings each, equal? %s" % (n, len(a), a == b))
    assert a == b

Parse counts at the **grammar** level.

In [ ]:
print("%-12s %-12s %-12s" % ("string", "ambiguous", "layered"))
for w in ['1', '1+2', '1+2*3', '1+2*3+1']:
    print("%-12s %-12d %-12d" % (w, nparses(Amb, w), nparses(Layer, w)))
    assert nparses(Layer, w) == 1

And at the **machine** level: accepting computations of the PDA.

In [ ]:
PA, PL = cfg2pda(Amb), cfg2pda(Layer)
print("%-10s %-14s %-14s" % ("string", "ambiguous PDA", "layered PDA"))
for w in ['1', '1+2', '1*2']:
    na = pda_npaths(PA, w, STKMAX=6)
    nl = pda_npaths(PL, w, STKMAX=8)
    print("%-10s %-14d %-14d" % (w, na, nl))
    assert nl == 1
assert pda_npaths(PA, '1+2*3', STKMAX=6) > 1

The counts agree with the parse-tree counts &mdash; computations **are** derivations.

In [ ]:
for w in ['1', '1+2', '1+2*3']:
    print("  %-8s trees %d, PDA computations %d"
          % (w, nparses(Amb, w), pda_npaths(PA, w, STKMAX=6)))
    assert nparses(Amb, w) == pda_npaths(PA, w, STKMAX=6)

Growth: the ambiguous machine's work multiplies, the layered one's does not.

In [ ]:
for w in ['1+2', '1+2*3', '1+2*3+1']:
    a = pda_npaths(PA, w, STKMAX=6)
    print("  %-10s ambiguous computations %2d, layered 1" % (w, a))
print("\nThe layered grammar is not merely tidier -- it is cheaper to parse,")
print("and it commits to one meaning.")

## 4. Animation

The layered goal machine: one computation per input.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimatePDA import *
AnimatePDA(cfg2pda(mkg({'E': ['T', 'E+T'], 'T': ['F', 'T*F'], 'F': ['1', '2', '(E)']}, 'E')), FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Plot computations against input length for the ambiguous grammar. What curve?
2. Why does one accepting computation correspond to one **leftmost** derivation?
3. Which is cheaper to check: ambiguity of a grammar, or of one string?

In [ ]:
# Your work for the exercises above.